In [1]:
import sys; sys.path.insert(0, "..")
from fantasy.yahoo_client import get_my_roster, get_free_agents, get_current_matchup, get_league_categories, get_roster_by_team
from fantasy.nba_stats import get_player_stats, get_games_this_week
from fantasy.analysis import project_team_categories, project_player, classify_categories
from fantasy.llm import build_waiver_prompt, ask_gemini
import pandas as pd

In [2]:
matchup = get_current_matchup()
week_start, week_end = matchup["week_start"], matchup["week_end"]
my_roster = get_my_roster()
opp_roster = get_roster_by_team(matchup["opponent_team_key"])
categories = get_league_categories()

def enrich(roster):
    return [{**p, "stats": get_player_stats(p["name"]),
             "games_remaining": get_games_this_week(p["team_abbr"], week_start, week_end)}
            for p in roster]

my_players = enrich(my_roster)
opp_players = enrich(opp_roster)

my_totals = project_team_categories(my_players)
opp_totals = project_team_categories(opp_players)

def to_league_cats(proj, cats):
    """Map raw stat projections to Yahoo league scoring category values."""
    result = {}
    for cat in cats:
        if cat in proj:
            result[cat] = proj[cat]
        elif cat == "FG%":
            result[cat] = proj["FGM"] / proj["FGA"] if proj.get("FGA", 0) > 0 else 0.0
        elif cat == "FT%":
            result[cat] = proj["FTM"] / proj["FTA"] if proj.get("FTA", 0) > 0 else 0.0
        elif cat == "3PTM":
            result[cat] = proj.get("FG3M", 0.0)
        elif cat == "ST":
            result[cat] = proj.get("STL", 0.0)
        elif cat == "TO":
            result[cat] = proj.get("TOV", 0.0)
        elif cat == "FGM/A":
            result[cat] = proj.get("FGM", 0.0)
        elif cat == "FTM/A":
            result[cat] = proj.get("FTM", 0.0)
    return result

my_cats = to_league_cats(my_totals, categories)
opp_cats = to_league_cats(opp_totals, categories)
classification = classify_categories(my_cats, opp_cats)

# Weaknesses = categories we are losing or in tossup
weak_cats = [cat for cat, status in classification.items() if status in ("loss", "tossup")]
print(f"Weakest categories: {weak_cats}")
print(f"\nMatchup breakdown:")
for cat, status in classification.items():
    print(f"  {cat:<8} {my_cats.get(cat,0):.2f} vs {opp_cats.get(cat,0):.2f}  → {status}")

[2026-03-24 00:29:26,022 DEBUG] [yahoo_oauth.oauth.__init__] Checking 
[2026-03-24 00:29:26,026 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 3265.348504781723
[2026-03-24 00:29:26,027 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID
[2026-03-24 00:29:26,032 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 3265.3546171188354
[2026-03-24 00:29:26,033 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID
[2026-03-24 00:29:26,925 DEBUG] [yahoo_oauth.oauth.__init__] Checking 
[2026-03-24 00:29:26,930 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 3266.252876996994
[2026-03-24 00:29:26,931 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID
[2026-03-24 00:29:26,938 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 3266.2609992027283
[2026-03-24 00:29:26,939 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID


Weakest categories: ['TO', 'BLK', 'ST']


In [3]:
# Typical weekly contribution per player (used to normalize fit scores)
TYPICAL_WEEKLY = {
    "PTS": 55,    # ~18ppg × 3g
    "REB": 18,    # ~6rpg × 3g
    "AST": 12,    # ~4apg × 3g
    "3PTM": 5,    # ~1.7 per game × 3g
    "ST": 3,      # ~1 spg × 3g
    "BLK": 2,     # ~0.7 bpg × 3g
    "TO": 6,      # ~2 topg × 3g
    "FGM/A": 14,
    "FTM/A": 8,
    "FG%": 0.47,
    "FT%": 0.78,
}

fas = get_free_agents(count=200)

scored = []
for fa in fas:
    stats = get_player_stats(fa["name"])
    games = get_games_this_week(fa["team_abbr"], week_start, week_end)
    if stats is None:
        continue
    proj = project_player(stats, games, fa.get("status", ""))
    fa_cats = to_league_cats(proj, categories)
    # Fit score: normalized contribution in weak categories (TOV inverted)
    fit = 0.0
    for c in weak_cats:
        typical = TYPICAL_WEEKLY.get(c, 1.0)
        val = fa_cats.get(c, 0)
        if c in ("TO", "TOV"):
            fit += (typical - val) / typical  # reward low TOV
        else:
            fit += val / typical
    scored.append({**fa, "games_remaining": games, "projected": proj, "fit_score": fit})

scored.sort(key=lambda x: x["fit_score"], reverse=True)
print(pd.DataFrame([{
    "Player": p["name"], "Pos": p["position"], "Status": p["status"] or "Active",
    "Games": p["games_remaining"], "Fit Score": round(p["fit_score"], 2)
} for p in scored[:50]]).to_string(index=False))

           Player      Pos Status  Games  Fit Score
      Brook Lopez        C Active      4       1.08
   DeAndre Jordan        C Active      4       1.06
      Paul George SG,SF,PF    GTD      3       1.04
      Eric Gordon    SG,SF     NA      4       1.03
    Nicolas Batum    SF,PF Active      4       1.03
       Taj Gibson     PF,C Active      4       1.02
    Klay Thompson    SG,SF Active      3       1.02
       Kevin Love     PF,C Active      4       1.02
      Mike Conley       PG Active      2       1.01
       Chris Paul       PG     NA      4       1.01
Jonas Valančiūnas        C Active      4       1.01
       Kyle Lowry       PG Active      3       1.01
       Al Horford     PF,C    INJ      4       1.00
       Jeff Green       PF Active      4       1.00
   Garrett Temple    PG,SG Active      4       1.00
  Bismack Biyombo        C Active      3       1.00


In [4]:
import importlib, os
from dotenv import load_dotenv
load_dotenv("/mnt/c/Users/louis/Documents/dev/nba_fantasy/.env", override=True)

prompt = build_waiver_prompt(scored, weak_cats)
advice = ask_gemini(prompt)
print("\n=== WAIVER WIRE PICKUPS ===\n")
print(advice)


=== WAIVER WIRE PICKUPS ===

Okay, based on your team's needs (TO, BLK, ST) and the available free agents, here are my top 5 recommendations:

1.  **Brook Lopez:** Lopez is the clear standout, offering elite blocks with excellent steals for a center, while keeping turnovers manageable given his production.
2.  **DeAndre Jordan:** While his scoring is low, Jordan's rebounding and blocks will significantly bolster your team's interior defense and rebounding, mitigating the negative impact of his turnovers.
3.  **Paul George:** George provides a well-rounded boost in steals, and blocks for a wing, complementing it with a decent amount of defensive stats.
4.  **Eric Gordon:** Gordon has a good source of steals for a guard-eligible player.
5.  **Nicolas Batum:** Batum is a versatile player who contributes across the board, including steals and acceptable blocks, while keeping his turnovers relatively low.


In [5]:
print(prompt)

You are a fantasy basketball analyst. My team is weak in: TO, BLK, ST.

Top available free agents scored against my needs:

Rank  Player                 Pos    G   FitScore   PTS    REB    AST    3PM    STL    BLK    TO    
-------------------------------------------------------------------------------------
1     Brook Lopez            C      4   1.08       38.5   15.3   7.0    6.3    2.7    4.4    3.5   
2     DeAndre Jordan         C      4   1.06       16.1   26.7   3.8    0.0    1.5    3.8    3.9   
3     Paul George            SG,SF,PF 3   1.04       36.0   11.5   8.3    5.4    3.4    1.1    3.6   
4     Eric Gordon            SG,SF  4   1.03       22.0   1.2    2.0    5.2    2.8    0.8    2.0   
5     Nicolas Batum          SF,PF  4   1.03       13.2   9.6    4.2    4.4    2.2    0.6    1.2   
6     Taj Gibson             PF,C   4   1.02       10.4   10.4   0.8    1.6    0.8    0.8    0.0   
7     Klay Thompson          SG,SF  3   1.02       38.8   4.3    4.0    9.8    1.5    0.

In [6]:
advice = ask_gemini(prompt + "how about moussa diabate")


In [8]:
print(advice)

Okay, based on your need for blocks (BLK) and steals (STL) while minimizing turnovers (TO), here are my top 5 recommendations from the free agent list, along with my reasoning, and then some info on Moussa Diabate:

**Top 5 Pickups:**

1.  **Brook Lopez:** Lopez is your absolute best option. Not only is he the highest-ranked FA, but he is a BLK and STL stud.

2.  **DeAndre Jordan:** Provides elite rebounding and strong blocks, while turnovers are higher than ideal, the defensive upside is too high to pass.

3.  **Paul George:** A strong overall contributor with exceptional steals.

4.  **Eric Gordon:** Can provide 3PM and PTS, but also contribute towards STLs.

5.  **Nicolas Batum:** Provides a good mix of defensive stats (STL) with less of a negative impact on turnovers.

**Moussa Diabate**

Moussa Diabate is currently signed with the Los Angeles Clippers on a two way contract. He has played a total of 73 games, averaging 3.2 PPG, 2.8 REB, 0.4 BLK and 0.2 STL. Considering this, Moussa